# M2 Hadoop WordCount Practice — PC / VS Code Notebook

This notebook is designed to run **line by line in VS Code** on a Windows PC.

It does **not** use Google Colab code such as:

```python
from google.colab import drive
!apt-get install ...
%%bash
```

## Goal

You will:

1. Set Java and Hadoop paths.
2. Create a small text file named `student_text.txt`.
3. Run Hadoop WordCount only on that file.
4. View the WordCount output.
5. Practice by editing the file and rerunning WordCount.


## Step 1 — Import libraries

In [1]:
import os
import shutil
import subprocess
from pathlib import Path

## Step 2 — Set Java and Hadoop paths

Update these two paths if your PC uses different installation folders.


In [2]:
from pathlib import Path
import os
import shutil
import subprocess
from collections import Counter
import re

# ============================================================
# Step 2 — Configure Java and Hadoop paths
# ============================================================
# The previous error happened because this file was missing:
# C:\hadoop-3.4.1\bin\hadoop.cmd
#
# This version tries common Hadoop install folders automatically.
# If your Hadoop is in a different folder, edit HADOOP_HOME manually below.

JAVA_HOME = r"C:\Program Files\Java\jdk-11"

HADOOP_VERSION = "3.4.1"
HADOOP_CANDIDATES = [
    r"C:\hadoop-3.4.1",
    r"C:\hadoop",
    r"C:\Program Files\hadoop-3.4.1",
]

HADOOP_HOME = None
for candidate in HADOOP_CANDIDATES:
    if (Path(candidate) / "bin" / "hadoop.cmd").exists():
        HADOOP_HOME = candidate
        break

# If auto-detection did not find Hadoop, keep the usual path.
# The notebook will still create WordCount output with a Python fallback.
if HADOOP_HOME is None:
    HADOOP_HOME = r"C:\hadoop-3.4.1"

os.environ["JAVA_HOME"] = JAVA_HOME
os.environ["HADOOP_HOME"] = HADOOP_HOME
os.environ["PATH"] = (
    str(Path(HADOOP_HOME) / "bin")
    + os.pathsep
    + str(Path(HADOOP_HOME) / "sbin")
    + os.pathsep
    + os.environ.get("PATH", "")
)

print("JAVA_HOME:", os.environ["JAVA_HOME"])
print("HADOOP_HOME:", os.environ["HADOOP_HOME"])


JAVA_HOME: C:\Program Files\Java\jdk-11
HADOOP_HOME: C:\hadoop-3.4.1


## Step 3 — Define working folders and Hadoop command paths

In [3]:
USER_HOME = Path.home()

INPUT_DIR = USER_HOME / "student_wordcount_input"
OUTPUT_DIR = USER_HOME / "student_wordcount_output"
STUDENT_FILE = INPUT_DIR / "student_text.txt"

HADOOP_CMD = Path(HADOOP_HOME) / "bin" / "hadoop.cmd"

MAPREDUCE_EXAMPLES_JAR = (
    Path(HADOOP_HOME)
    / "share"
    / "hadoop"
    / "mapreduce"
    / f"hadoop-mapreduce-examples-{HADOOP_VERSION}.jar"
)

print("Input folder:", INPUT_DIR)
print("Output folder:", OUTPUT_DIR)
print("Student file:", STUDENT_FILE)
print("Hadoop command:", HADOOP_CMD)
print("MapReduce examples jar:", MAPREDUCE_EXAMPLES_JAR)


Input folder: /Users/yuzhang/student_wordcount_input
Output folder: /Users/yuzhang/student_wordcount_output
Student file: /Users/yuzhang/student_wordcount_input/student_text.txt
Hadoop command: C:\hadoop-3.4.1/bin/hadoop.cmd
MapReduce examples jar: C:\hadoop-3.4.1/share/hadoop/mapreduce/hadoop-mapreduce-examples-3.4.1.jar


## Step 4 — Check installation

Run this cell to confirm Java and Hadoop are available.


In [4]:
from pathlib import Path

print("Checking paths...")

# Check JAVA_HOME
java_path = Path(JAVA_HOME)

if java_path.exists():
    print(f"[OK] JAVA_HOME found: {JAVA_HOME}")
else:
    print(f"[WARNING] JAVA_HOME does not exist: {JAVA_HOME}")

# Check HADOOP_HOME
hadoop_path = Path(HADOOP_HOME)

if hadoop_path.exists():
    print(f"[OK] HADOOP_HOME found: {HADOOP_HOME}")
else:
    print(f"[WARNING] HADOOP_HOME does not exist: {HADOOP_HOME}")
    print("Please verify your Hadoop installation path.")

# Check Hadoop command
if HADOOP_CMD.exists():
    print(f"[OK] Hadoop command found: {HADOOP_CMD}")
else:
    print(f"[WARNING] Hadoop command not found: {HADOOP_CMD}")

# Check MapReduce examples jar
if MAPREDUCE_EXAMPLES_JAR.exists():
    print(f"[OK] MapReduce examples jar found:")
    print(MAPREDUCE_EXAMPLES_JAR)
else:
    print("[WARNING] MapReduce examples jar not found:")
    print(MAPREDUCE_EXAMPLES_JAR)

print("\nPath validation completed.")

Checking paths...
[WARNING] JAVA_HOME does not exist: C:\Program Files\Java\jdk-11
[WARNING] HADOOP_HOME does not exist: C:\hadoop-3.4.1
Please verify your Hadoop installation path.
[WARNING] Hadoop command not found: C:\hadoop-3.4.1/bin/hadoop.cmd
[WARNING] MapReduce examples jar not found:
C:\hadoop-3.4.1/share/hadoop/mapreduce/hadoop-mapreduce-examples-3.4.1.jar

Path validation completed.


## Step 5 — Helper function to run terminal commands

This lets the notebook run Hadoop commands from Python.


In [5]:
def run_command(command, allow_fail=False):
    print("-" * 60)
    print("Running command:")
    print(" ".join(map(str, command)))
    print("-" * 60)

    result = subprocess.run(
        command,
        text=True,
        capture_output=True,
        env=os.environ.copy()
    )

    if result.stdout:
        print("STDOUT:")
        print(result.stdout)

    if result.stderr:
        print("STDERR:")
        print(result.stderr)

    if result.returncode != 0 and not allow_fail:
        raise RuntimeError(f"Command failed with return code {result.returncode}")

    return result


def python_wordcount(input_file, output_dir):
    """Fallback WordCount so Step 9 always has an output file."""
    output_dir.mkdir(parents=True, exist_ok=True)
    text = input_file.read_text(encoding="utf-8", errors="ignore").lower()
    words = re.findall(r"[A-Za-z0-9']+", text)
    counts = Counter(words)

    output_file = output_dir / "part-00000"
    output_file.write_text(
        "\n".join(f"{word}\t{count}" for word, count in sorted(counts.items())) + "\n",
        encoding="utf-8"
    )

    print("[OK] Python fallback WordCount completed.")
    print("Output file created:", output_file)
    return output_file


## Step 6 — Verify Java and Hadoop

In [6]:
# Step 5 — Check Java and Hadoop installation

def check_command(command):
    print("Running command:")
    print(" ".join(map(str, command)))
    print("-" * 60)

    try:
        result = subprocess.run(
            command,
            text=True,
            capture_output=True,
            env=os.environ.copy()
        )

        if result.stdout:
            print("STDOUT:")
            print(result.stdout)

        if result.stderr:
            print("STDERR:")
            print(result.stderr)

        if result.returncode == 0:
            print("[OK] Command completed successfully.")
        else:
            print(f"[WARNING] Command exited with code {result.returncode}.")

    except FileNotFoundError as e:
        print("[ERROR] Command not found.")
        print(f"Missing file: {e.filename}")
        print("Please check the path or installation.")


# Check Java
java_exe = shutil.which("java")

if java_exe:
    check_command(["java", "-version"])
else:
    print("[WARNING] Java command not found. Install Java or fix JAVA_HOME/PATH.")


# Check Hadoop
if HADOOP_CMD.exists():
    check_command([str(HADOOP_CMD), "version"])
else:
    print("[WARNING] Hadoop command not found:")
    print(HADOOP_CMD)
    print()
    print("This is why the old notebook produced no output file.")
    print("Step 8 will now use a Python fallback WordCount so Step 9 can display results.")
    print()
    print("To use real Hadoop later, install Hadoop or edit HADOOP_HOME to the correct folder.")


Running command:
java -version
------------------------------------------------------------
STDERR:
java version "24.0.2" 2025-07-15
Java(TM) SE Runtime Environment (build 24.0.2+12-54)
Java HotSpot(TM) 64-Bit Server VM (build 24.0.2+12-54, mixed mode, sharing)

[OK] Command completed successfully.
[WARNING] Hadoop command not found:
C:\hadoop-3.4.1/bin/hadoop.cmd

This is why the old notebook produced no output file.
Step 8 will now use a Python fallback WordCount so Step 9 can display results.

To use real Hadoop later, install Hadoop or edit HADOOP_HOME to the correct folder.


## Step 7 — Create a clean input folder and student text file

This cell creates **one small input file** only.

This avoids accidentally reading a large folder.


In [7]:
# Clean old input and output folders
if INPUT_DIR.exists():
    shutil.rmtree(INPUT_DIR)

if OUTPUT_DIR.exists():
    shutil.rmtree(OUTPUT_DIR)

INPUT_DIR.mkdir(parents=True, exist_ok=True)

student_text = """Hadoop helps count words
Hadoop works with big data
Students practice Hadoop WordCount
Data data data is everywhere
WordCount counts each word
"""

STUDENT_FILE.write_text(student_text, encoding="utf-8")

print("Student file created:")
print(STUDENT_FILE)

print("\\nStudent file content:")
print(STUDENT_FILE.read_text(encoding="utf-8"))

Student file created:
/Users/yuzhang/student_wordcount_input/student_text.txt
\nStudent file content:
Hadoop helps count words
Hadoop works with big data
Students practice Hadoop WordCount
Data data data is everywhere
WordCount counts each word



## Step 8 — Run Hadoop WordCount on only `student_text.txt`

Important: this reads only:

```text
student_text.txt
```

It does **not** read the whole input folder.


In [ ]:
# # Step 8 — Run WordCount
# # This first tries real Hadoop. If Hadoop is not installed/found,
# # it automatically creates the same style output with Python.

# if OUTPUT_DIR.exists():
#     shutil.rmtree(OUTPUT_DIR)

# if HADOOP_CMD.exists() and MAPREDUCE_EXAMPLES_JAR.exists():
#     wordcount_command = [
#         str(HADOOP_CMD),
#         "jar",
#         str(MAPREDUCE_EXAMPLES_JAR),
#         "wordcount",
#         str(STUDENT_FILE),
#         str(OUTPUT_DIR)
#     ]

#     result = run_command(wordcount_command, allow_fail=True)

#     if result.returncode != 0:
#         print("[WARNING] Hadoop failed. Running Python fallback instead.")
#         python_wordcount(STUDENT_FILE, OUTPUT_DIR)
# else:
#     print("[WARNING] Hadoop files were not found.")
#     print("Missing Hadoop command:", HADOOP_CMD)
#     print("Missing example jar:", MAPREDUCE_EXAMPLES_JAR)
#     print("Running Python fallback instead.")
#     python_wordcount(STUDENT_FILE, OUTPUT_DIR)


[WARNING] Hadoop files were not found.
Missing Hadoop command: C:\hadoop-3.4.1/bin/hadoop.cmd
Missing example jar: C:\hadoop-3.4.1/share/hadoop/mapreduce/hadoop-mapreduce-examples-3.4.1.jar
Running Python fallback instead.
[OK] Python fallback WordCount completed.
Output file created: /Users/yuzhang/student_wordcount_output/part-00000


## Step 9 — Display WordCount output

In [ ]:
# # Step 9 — Display WordCount output

# output_files = sorted(OUTPUT_DIR.glob("part-*"))

# if not output_files:
#     print("No output file found.")
#     print("Please rerun Step 7, then Step 8.")
# else:
#     for output_file in output_files:
#         print("Output file:", output_file)
#         print("-" * 60)
#         print(output_file.read_text(encoding="utf-8", errors="ignore"))


Output file: /Users/yuzhang/student_wordcount_output/part-00000
------------------------------------------------------------
big	1
count	1
counts	1
data	4
each	1
everywhere	1
hadoop	3
helps	1
is	1
practice	1
students	1
with	1
word	1
wordcount	2
words	1
works	1



# Student Practice

Now students should edit the input file and rerun WordCount.

## Practice instructions

1. Open this file on your PC:

```text
student_wordcount_input/student_text.txt
```

2. Replace the text with your own 5 or more lines.

Example idea:

```text
People in those industries often travel globally.
They move between projects.
They work with contractors and business groups.
Some periods are very intense.
Other periods are quieter.
```

3. Rerun:

- Step 8
- Step 9

4. Answer:

- Which word appears most often?
- Did the output change?
- Why did the count change?
- What is the difference between running WordCount on one file versus the entire folder?


## Optional Practice Cell — Update `student_text.txt` from inside the notebook

Students may use this cell instead of manually opening the file.

They can change the text below, then rerun Step 8 and Step 9.


In [ ]:
# new_student_text = """People in those industries often travel globally
# They move between projects
# They work with multiple contractors and business groups
# Some periods are very intense
# Other periods are quieter
# """

# STUDENT_FILE.write_text(new_student_text, encoding="utf-8")

# print("Updated student_text.txt:")
# print(STUDENT_FILE.read_text(encoding="utf-8"))

Updated student_text.txt:
People in those industries often travel globally
They move between projects
They work with multiple contractors and business groups
Some periods are very intense
Other periods are quieter

